In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [3]:
# Load the data
age_analysis_df = pd.read_excel('Data/Age Analysis.xlsx')

csv_folder = os.path.join(os.getcwd(), "csv_outputs")
json_folder = os.path.join(os.getcwd(), "json_outputs")

In [4]:
print("Columns:", age_analysis_df.columns.tolist())

Columns: ['CUSTOMER_NUMBER', 'FIN_PERIOD', 'TOTAL_DUE', 'AMT_CURRENT', 'AMT_30_DAYS', 'AMT_60_DAYS', 'AMT_90_DAYS', 'AMT_120_DAYS', 'AMT_150_DAYS', 'AMT_180_DAYS', 'AMT_210_DAYS', 'AMT_240_DAYS', 'AMT_270_DAYS', 'AMT_300_DAYS', 'AMT_330_DAYS', 'AMT_360_DAYS']


In [5]:
print("Sample data:\n", age_analysis_df.head())

Sample data:
   CUSTOMER_NUMBER  FIN_PERIOD  TOTAL_DUE  AMT_CURRENT  AMT_30_DAYS  \
0          AACJ01      201810     456.00       456.00          0.0   
1          AACJ01      201811     456.00         0.00        456.0   
2          AACJ01      201812     456.00         0.00          0.0   
3          AACJ01      201901     456.00         0.00          0.0   
4          AACJ01      201902     463.32         7.32          0.0   

   AMT_60_DAYS  AMT_90_DAYS  AMT_120_DAYS  AMT_150_DAYS  AMT_180_DAYS  \
0          0.0          0.0           0.0           0.0             0   
1          0.0          0.0           0.0           0.0             0   
2        456.0          0.0           0.0           0.0             0   
3          0.0        456.0           0.0           0.0             0   
4          0.0          0.0         456.0           0.0             0   

   AMT_210_DAYS  AMT_240_DAYS  AMT_270_DAYS  AMT_300_DAYS  AMT_330_DAYS  \
0             0             0             0        

In [6]:
print("Null values:\n", age_analysis_df.isnull().sum())

Null values:
 CUSTOMER_NUMBER    0
FIN_PERIOD         0
TOTAL_DUE          0
AMT_CURRENT        0
AMT_30_DAYS        0
AMT_60_DAYS        0
AMT_90_DAYS        0
AMT_120_DAYS       0
AMT_150_DAYS       0
AMT_180_DAYS       0
AMT_210_DAYS       0
AMT_240_DAYS       0
AMT_270_DAYS       0
AMT_300_DAYS       0
AMT_330_DAYS       0
AMT_360_DAYS       0
dtype: int64


In [7]:
# Remove rows with any null values
customer_parameters_df = age_analysis_df.dropna()

In [8]:
print("Data shape:", age_analysis_df.shape)

Data shape: (21914, 16)


In [9]:
age_analysis_df

,CUSTOMER_NUMBER,FIN_PERIOD,TOTAL_DUE,AMT_CURRENT,AMT_30_DAYS,AMT_60_DAYS,AMT_90_DAYS,AMT_120_DAYS,AMT_150_DAYS,AMT_180_DAYS,AMT_210_DAYS,AMT_240_DAYS,AMT_270_DAYS,AMT_300_DAYS,AMT_330_DAYS,AMT_360_DAYS
0,AACJ01,201810,456.00,456.00,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
1,AACJ01,201811,456.00,0.00,456.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
2,AACJ01,201812,456.00,0.00,0.0,456.0,0.0,0.0,0.0,0,0,0,0,0,0,0
3,AACJ01,201901,456.00,0.00,0.0,0.0,456.0,0.0,0.0,0,0,0,0,0,0,0
4,AACJ01,201902,463.32,7.32,0.0,0.0,0.0,456.0,0.0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21909,ZNAO01,201908,-116.27,-116.27,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
21910,ZNAO01,201909,1023.73,1023.73,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
21911,ZNAO01,201910,672.60,672.60,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0
21912,ZNAO01,201911,172.60,0.00,172.6,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0


In [ ]:
age_analysis_df['CUSTOMER_NUMBER'] = age_analysis_df['CUSTOMER_NUMBER'].astype(str).str.strip()

age_analysis_df['FIN_PERIOD'] = pd.to_datetime(age_analysis_df['FIN_PERIOD'], format='%Y%m')

amt_cols = [col for col in age_analysis_df.columns if 'AMT_' in col or 'TOTAL_DUE' in col]
age_analysis_df[amt_cols] = age_analysis_df[amt_cols].apply(pd.to_numeric, errors='coerce')

age_analysis_df['TOTAL_DUE'] = age_analysis_df[amt_cols].sum(axis=1)

# Identify all AMT_x_DAYS columns
amt_days_cols = [col for col in age_analysis_df.columns if re.match(r'AMT_\d+_DAYS', col)]

# Combine into a JSON string per row for Power BI compatibility
age_analysis_df['AMT_DAYS_BUCKETS'] = age_analysis_df[amt_days_cols].apply(lambda row: json.dumps(row.dropna().to_dict()), axis=1)

# Drop the original columns
age_analysis_df = age_analysis_df.drop(columns=amt_days_cols)


In [11]:
age_analysis_df

,CUSTOMER_NUMBER,FIN_PERIOD,TOTAL_DUE,AMT_CURRENT,AMT_DAYS_BUCKETS
0,AACJ01,2018-10-01,912.00,456.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
1,AACJ01,2018-11-01,912.00,0.00,"{'AMT_30_DAYS': 456.0, 'AMT_60_DAYS': 0.0, 'AM..."
2,AACJ01,2018-12-01,912.00,0.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 456.0, 'AM..."
3,AACJ01,2019-01-01,912.00,0.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
4,AACJ01,2019-02-01,926.64,7.32,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
...,...,...,...,...,...
21909,ZNAO01,2019-08-01,-232.54,-116.27,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21910,ZNAO01,2019-09-01,2047.46,1023.73,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21911,ZNAO01,2019-10-01,1345.20,672.60,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21912,ZNAO01,2019-11-01,345.20,0.00,"{'AMT_30_DAYS': 172.6, 'AMT_60_DAYS': 0.0, 'AM..."


In [12]:
#age_analysis_df = age_analysis_df[['CUSTOMER_NUMBER', 'FIN_PERIOD', 'TOTAL_DUE', 'AMT_CURRENT']]
age_analysis_df.columns.tolist()

['CUSTOMER_NUMBER',
 'FIN_PERIOD',
 'TOTAL_DUE',
 'AMT_CURRENT',
 'AMT_DAYS_BUCKETS']

In [13]:
age_analysis_df

,CUSTOMER_NUMBER,FIN_PERIOD,TOTAL_DUE,AMT_CURRENT,AMT_DAYS_BUCKETS
0,AACJ01,2018-10-01,912.00,456.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
1,AACJ01,2018-11-01,912.00,0.00,"{'AMT_30_DAYS': 456.0, 'AMT_60_DAYS': 0.0, 'AM..."
2,AACJ01,2018-12-01,912.00,0.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 456.0, 'AM..."
3,AACJ01,2019-01-01,912.00,0.00,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
4,AACJ01,2019-02-01,926.64,7.32,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
...,...,...,...,...,...
21909,ZNAO01,2019-08-01,-232.54,-116.27,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21910,ZNAO01,2019-09-01,2047.46,1023.73,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21911,ZNAO01,2019-10-01,1345.20,672.60,"{'AMT_30_DAYS': 0.0, 'AMT_60_DAYS': 0.0, 'AMT_..."
21912,ZNAO01,2019-11-01,345.20,0.00,"{'AMT_30_DAYS': 172.6, 'AMT_60_DAYS': 0.0, 'AM..."


In [14]:
age_analysis_df.to_csv(os.path.join(csv_folder, "age_analysis_clean.csv"), index=False)
age_analysis_df.to_json(os.path.join(json_folder, "age_analysis_clean.json"), orient="records", lines=True)

CUSTOMER SEGMENTATION AND GROUPING

In [15]:
#Group by customer number and financial period
#customer_segment = age_analysis_df.groupby(['CUSTOMER_NUMBER', 'FIN_PERIOD'])[amt_cols].sum()

#customer_segment

In [16]:
#customer_segment = customer_segment.reset_index()

### Investigating Customer Variants

This section explores whether visually similar customer numbers (e.g., AACJ01 vs AACJC2) reflect sub-accounts, branches, or internal segmentation.

We extract prefixes and suffixes to identify potential groupings, then analyze whether customers sharing a prefix exhibit similar financial behavior across aging buckets.

This helps inform whether to roll up these variants in reporting or treat them as distinct entities.

In [17]:
# Create a copy
customer_number_df = age_analysis_df[['CUSTOMER_NUMBER']].drop_duplicates().copy()

# Extract patterns
customer_number_df['prefix'] = customer_number_df['CUSTOMER_NUMBER'].str.extract(r'^([A-Z]{1,4})')
customer_number_df['suffix'] = customer_number_df['CUSTOMER_NUMBER'].str.extract(r'(\d+)$')
customer_number_df['variant'] = customer_number_df['CUSTOMER_NUMBER'].str.extract(r'([A-Z]+\d+)$')

# Merge back to get financial metrics
merged_df = age_analysis_df.merge(customer_number_df, on='CUSTOMER_NUMBER', how='left')

# Group by prefix to explore behavioral similarity
#grouped_metrics = merged_df.groupby('prefix')[amt_cols].mean().round(2)

#grouped_metrics

In [18]:
#Flag potential variants

# Create a working copy
customer_number_var_df = customer_number_df.copy()

# Extract base code (first 4 letters)
customer_number_var_df['base_code'] = customer_number_var_df['CUSTOMER_NUMBER'].str.extract(r'^([A-Z]{4})')

# Flag potential variants based on shared base code
customer_number_var_df['is_variant'] = customer_number_var_df.duplicated('base_code', keep=False)

# Display result
customer_number_var_df

,CUSTOMER_NUMBER,prefix,suffix,variant,base_code,is_variant
0,AACJ01,AACJ,01,AACJ01,AACJ,True
9,AACJC2,AACJ,2,AACJC2,AACJ,True
15,AADPRG,AADP,NaN,NaN,AADP,True
37,AADPRO,AADP,NaN,NaN,AADP,True
60,AAMI01,AAMI,01,AAMI01,AAMI,False
...,...,...,...,...,...,...
21827,ZDOE01,ZDOE,01,ZDOE01,ZDOE,False
21840,ZHAY02,ZHAY,02,ZHAY02,ZHAY,False
21863,ZMAU01,ZMAU,01,ZMAU01,ZMAU,False
21869,ZNAE01,ZNAE,01,ZNAE01,ZNAE,False


### Due Amount Volatility

This block computes the standard deviation of `TOTAL_DUE` across periods for each customer.

High volatility may indicate erratic payment behavior, seasonal spikes, or inconsistent invoicing. It’s useful for identifying unstable accounts.

In [19]:
volatility_df = age_analysis_df.groupby('CUSTOMER_NUMBER')['TOTAL_DUE'].std().reset_index(name='DUE_VOLATILITY')
volatility_df

,CUSTOMER_NUMBER,DUE_VOLATILITY
0,'DGSOC,0.000000
1,AACJ01,28.395473
2,AACJC2,6013.497319
3,AADPRG,230861.022611
4,AADPRO,66303.524981
...,...,...
1663,ZDOE01,4160.937806
1664,ZHAY02,9128.714849
1665,ZMAU01,3645.619652
1666,ZNAE01,27394.025575


### Negative Balance Detection

This block filters rows where `TOTAL_DUE` is negative, which may indicate refunds, reversals, or data entry errors.

Flagging these rows is essential for audit integrity and downstream calculations. Negative balances can distort overdue ratios and risk scores.

In [20]:
negative_due_df = age_analysis_df[age_analysis_df['TOTAL_DUE'] < 0]
negative_due_df[['CUSTOMER_NUMBER', 'FIN_PERIOD', 'TOTAL_DUE']]

,CUSTOMER_NUMBER,FIN_PERIOD,TOTAL_DUE
84,ABHN01,2019-04-01,-1641.60
85,ABHN01,2019-05-01,-1641.60
86,ABHN01,2019-06-01,-1641.60
87,ABHN01,2019-07-01,-1641.60
88,ABHN01,2019-08-01,-1641.60
...,...,...,...
21903,ZNAO01,2019-02-01,-1.54
21904,ZNAO01,2019-03-01,-1.54
21907,ZNAO01,2019-06-01,-232.54
21908,ZNAO01,2019-07-01,-232.54
